In [1]:
import joblib
import pandas as pd
import numpy as np
import requests

knn = joblib.load("knn_model.pkl")

print("KNN loaded successfully!")
print("Features:", knn.feature_names_in_)
print("Neighbors:", knn[-1].n_neighbors)

KNN loaded successfully!
Features: ['Market Share' 'Passenger Load Factor' 'OTP Index'
 'Hourly_Airport_Departures' 'Distance' 'Airport_Congestion_Index'
 'VC_2h_precip' 'VC_2h_temp']
Neighbors: 19


In [2]:
data = pd.read_csv("4820_FinalProjectCleaned.csv")

print(data.columns.tolist())

['Used Date', 'From', 'To', 'Airline', 'Scheduled Departure', 'SDEP', 'DEP', 'SARR', 'ARR', 'Departure Delay', 'Distance', 'Passenger Load Factor', 'Airline Rating', 'Airport Rating', 'Market Share', 'OTP Index', 'Category', 'Departure_Hour', 'Scheduled_DateTime', 'Weather_Cutoff', 'time', 'Hourly_Airport_Departures', 'Airport_Congestion_Index', 'Is_Weekend', 'Is_Delayed', 'Weather_Hour', 'VC_temp', 'VC_humidity', 'VC_precip', 'VC_windspeed', 'VC_cloudcover', 'VC_visibility', 'VC_conditions', 'Past_windspeedKmph', 'Past_precipMM', 'Weather_Hour_2h', 'VC_2h_temp', 'VC_2h_humidity', 'VC_2h_precip', 'VC_2h_windspeed', 'VC_2h_cloudcover', 'VC_2h_visibility', 'VC_2h_conditions']


In [3]:
features = knn.feature_names_in_

X_test = data[features].iloc[[0]].copy()
X_test = X_test.apply(pd.to_numeric, errors="coerce")

prediction = knn.predict(X_test)[0]
probabilities = knn.predict_proba(X_test)[0]

print("Prediction:", "Delayed" if prediction == 1 else "On Time")
print("Chance of delay:", f"{probabilities[1]:.2%}")

display(X_test)

Prediction: Delayed
Chance of delay: 52.63%


,Market Share,Passenger Load Factor,OTP Index,Hourly_Airport_Departures,Distance,Airport_Congestion_Index,VC_2h_precip,VC_2h_temp
0,3.8,84.2,76.2,7.0,1138,0.040698,0.0,18.4


In [4]:
from math import radians, sin, cos, sqrt, asin

airport_coords = {
    "BLR": (13.1986, 77.7066),
    "BOM": (19.0896, 72.8656),
    "CCU": (22.6547, 88.4467),
    "DEL": (28.5562, 77.1000),
    "MAA": (12.9941, 80.1709),
    "HYD": (17.2403, 78.4294),
    "GOI": (15.3808, 73.8314),
    "AMD": (23.0772, 72.6347)
}

def calculate_distance(origin, destination):
    lat1, lon1 = map(radians, airport_coords[origin])
    lat2, lon2 = map(radians, airport_coords[destination])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2)**2
        + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    )

    return 6371 * 2 * asin(sqrt(a))

In [ ]:
######
#DEMO#
######


# Display available departure airports
airports = sorted(data["From"].dropna().unique())

print("Available departure airports:")
print(", ".join(airports))

origin = input("Departure airport: ").strip().upper()

while origin not in airports:
    origin = input("Invalid airport. Try again: ").strip().upper()

# Display available destinations from selected airport
destinations = sorted(
    data.loc[data["From"] == origin, "To"].dropna().unique()
)

print("\nAvailable destinations:")
print(", ".join(destinations))

destination = input("Destination airport: ").strip().upper()

while destination not in destinations:
    destination = input("Invalid destination. Try again: ").strip().upper()

# Display airlines operating the selected route
route = data[
    (data["From"] == origin) &
    (data["To"] == destination)
]

airlines = sorted(route["Airline"].dropna().unique())

print("\nAvailable airlines:")
print(", ".join(airlines))

airline = input("Airline: ").strip()

while airline not in airlines:
    airline = input("Invalid airline. Try again: ").strip()

# Collect date and time
date = input("\nFlight date (YYYY-MM-DD): ")
time = input("Departure time (HH:MM IST): ")

departure = pd.Timestamp(f"{date} {time}")
weather_hour = departure.floor("h") - pd.Timedelta(hours=2)

print("\nFlight:", origin, "->", destination)
print("Airline:", airline)
print("Departure:", departure)
print("Required weather time:", weather_hour)

Available departure airports:
BLR, BOM, CCU, DEL

Available destinations:
HYD


In [8]:
import requests
from getpass import getpass

API_KEY = getpass("Enter Visual Crossing API key: ")

airport_locations = {
    "BLR": "13.1986,77.7066",
    "BOM": "19.0896,72.8656",
    "CCU": "22.6547,88.4467",
    "DEL": "28.5562,77.1000"
}

# Calculate both required weather hours
current_hour = departure.floor("h")
previous_hour = current_hour - pd.Timedelta(hours=2)

location = airport_locations[origin]

start_date = previous_hour.strftime("%Y-%m-%d")
end_date = current_hour.strftime("%Y-%m-%d")

url = (
    "https://weather.visualcrossing.com/"
    "VisualCrossingWebServices/rest/services/timeline/"
    f"{location}/{start_date}/{end_date}"
)

params = {
    "unitGroup": "metric",
    "include": "hours",
    "elements": "datetime,temp,humidity,precip,windspeed,visibility,cloudcover",
    "key": API_KEY,
    "contentType": "json"
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

weather_data = response.json()

# Combine hourly records
records = []

for day in weather_data["days"]:
    for hour in day["hours"]:
        records.append({
            **hour,
            "DateTime": pd.Timestamp(
                f"{day['datetime']} {hour['datetime']}"
            )
        })

hourly = pd.DataFrame(records).set_index("DateTime")

current = hourly.loc[current_hour]
previous = hourly.loc[previous_hour]

# Construct all 11 weather features
weather_features = {
    "VC_windspeed": current["windspeed"],
    "VC_cloudcover": current["cloudcover"],
    "VC_visibility": current["visibility"],
    "VC_temp": current["temp"],
    "VC_humidity": current["humidity"],
    "VC_precip": current["precip"],
    "VC_2h_temp": previous["temp"],
    "VC_2h_humidity": previous["humidity"],
    "VC_2h_precip": previous["precip"],
    "VC_2h_windspeed": previous["windspeed"],
    "VC_2h_visibility": previous["visibility"]
}

print("Weather retrieved successfully!")
print("Timezone:", weather_data.get("timezone"))

display(pd.DataFrame([weather_features]))

Weather retrieved successfully!
Timezone: Asia/Kolkata


,VC_windspeed,VC_cloudcover,VC_visibility,VC_temp,VC_humidity,VC_precip,VC_2h_temp,VC_2h_humidity,VC_2h_precip,VC_2h_windspeed,VC_2h_visibility
0,3.6,50.0,6.0,24.0,99.96,0.0,22.0,100.0,0.0,7.6,6.0


In [ ]:
# Select airline data
airline_data = data[data["Airline"] == airline]

# Select departure airport data
airport_data = data[data["From"] == origin]

# Select route data
route_data = data[
    (data["From"] == origin) &
    (data["To"] == destination)
]

# Select airport departures at the scheduled hour
hour_data = airport_data[
    airport_data["Departure_Hour"] == departure.hour
]

operational_features = {
    "Market Share": airline_data["Market Share"].median(),

    "Passenger Load Factor": 
        airline_data["Passenger Load Factor"].median(),

    "OTP Index": airline_data["OTP Index"].median(),

    "Hourly_Airport_Departures":
        hour_data["Hourly_Airport_Departures"].median(),

    "Distance": calculate_distance(origin, destination),

    "Airport_Congestion_Index":
        airport_data["Airport_Congestion_Index"].median()
}

display(pd.DataFrame([operational_features]))

,Market Share,Passenger Load Factor,OTP Index,Hourly_Airport_Departures,Distance,Airport_Congestion_Index
0,6.5,85.0,85.6,87.0,848.0,0.564935


In [10]:
X_full = pd.DataFrame([{
    **weather_features,
    **operational_features
}])

X_full = X_full.apply(pd.to_numeric, errors="coerce")

print("Input shape:", X_full.shape)
print("Missing values:", X_full.isna().sum().sum())

display(X_full)

Input shape: (1, 17)
Missing values: 0


,VC_windspeed,VC_cloudcover,VC_visibility,VC_temp,VC_humidity,VC_precip,VC_2h_temp,VC_2h_humidity,VC_2h_precip,VC_2h_windspeed,VC_2h_visibility,Market Share,Passenger Load Factor,OTP Index,Hourly_Airport_Departures,Distance,Airport_Congestion_Index
0,3.6,50.0,6.0,24.0,99.96,0.0,22.0,100.0,0.0,7.6,6.0,6.5,85.0,85.6,87.0,848.0,0.564935


In [11]:
X_knn = X_full[knn.feature_names_in_]

prediction = knn.predict(X_knn)[0]
probabilities = knn.predict_proba(X_knn)[0]

print("=" * 45)
print("FLIGHT DELAY PREDICTION")
print("=" * 45)

print(f"Route: {origin} -> {destination}")
print(f"Airline: {airline}")
print(f"Departure: {departure}")

print("\nModel: KNN")
print(f"Chance of delay: {probabilities[1]:.2%}")
print(f"Chance of no delay: {probabilities[0]:.2%}")

print("\nFINAL PREDICTION:",
      "DELAYED" if prediction == 1 else "ON TIME")

FLIGHT DELAY PREDICTION
Route: BLR -> BOM
Airline: Air Asia
Departure: 2020-07-23 08:24:00

Model: KNN
Chance of delay: 15.79%
Chance of no delay: 84.21%

FINAL PREDICTION: ON TIME
